# Neural Networks for Tabular Data — Entity Embeddings vs Gradient Boosting

Most real-world tabular data mixes **numeric** columns (age, income) with **categorical** columns (city, product, occupation). The classic way to feed categoricals to a model is **one-hot encoding**: turn a column with $C$ distinct values into $C$ binary columns. That works, but it is sparse, wasteful, and treats every category as equidistant from every other one.

**Entity embeddings** (popularized by the Kaggle Rossmann winners) replace one-hot with a small **learned dense vector per category**, exactly like a word embedding. The network learns, from the loss gradient, which categories behave similarly and places them near each other in the embedding space.

In this notebook we:

1. Build a **synthetic** tabular dataset in pandas where the binary target genuinely depends on *both* categorical and numeric columns (so embeddings actually matter).
2. Build a small **PyTorch** `nn.Module` with one `nn.Embedding` per categorical feature, concatenated with the numerics, followed by an MLP head.
3. Train it on CPU for a handful of epochs and report validation loss / AUC.
4. Compare head-to-head against `sklearn`'s `GradientBoostingClassifier` on one-hot features (test accuracy + ROC-AUC, printed and as a bar chart).

> Note on libraries: `xgboost` / `lightgbm` are **not** installed here, so we benchmark against scikit-learn's `GradientBoostingClassifier`. In a real project you would benchmark against **XGBoost** or **LightGBM**, which are far faster and usually the strongest tabular baselines.

In [ ]:
import numpy as np                      # data generation + array math
import pandas as pd                     # the tabular dataframe itself
import matplotlib.pyplot as plt         # final comparison bar chart

import torch                            # tensors + autograd
import torch.nn as nn                   # layers: Embedding, Linear, Dropout, ...
from torch.utils.data import TensorDataset, DataLoader  # mini-batching

from sklearn.model_selection import train_test_split    # train / val / test splits
from sklearn.preprocessing import StandardScaler        # scale numeric columns
from sklearn.ensemble import GradientBoostingClassifier # the GBM baseline
from sklearn.metrics import roc_auc_score, accuracy_score

# --- Reproducibility ---
# Seed every RNG we touch so the whole notebook is deterministic run-to-run.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)        # seeds torch's CPU RNG (weight init, dropout masks, shuffling)
rng = np.random.default_rng(SEED)  # modern NumPy Generator used to synthesize the data

# Keep everything on CPU: the model is tiny and this must run offline in seconds.
device = torch.device("cpu")
print("torch", torch.__version__, "| device:", device)

## 1. A synthetic tabular dataset

We hand-build a dataset so we *know* the ground truth and can guarantee the target depends on both kinds of feature. Columns:

| column | type | cardinality | role |
|---|---|---|---|
| `city` | categorical (string) | 8 | each city carries a hidden risk weight |
| `device` | categorical (string) | 4 | each device carries a hidden risk weight |
| `plan` | categorical (string) | 5 | each plan carries a hidden risk weight |
| `age` | numeric | – | continuous driver |
| `tenure` | numeric | – | continuous driver |
| `balance` | numeric | – | continuous driver |
| `churn` | **binary target** | – | 1 = customer churns |

**Data-generating process.** We assign each *category value* a hidden latent weight, form a linear logit

$$z = w_{\text{city}} + w_{\text{device}} + w_{\text{plan}} + \beta^{\top}\mathbf{x}_{\text{num}} + (\text{interaction}) + \varepsilon,$$

then draw the label from $y \sim \text{Bernoulli}(\sigma(z))$. Because the categorical weights differ and there is a city×numeric interaction, a model must actually *understand the categories* to predict well — pure numerics are not enough.

In [ ]:
N = 8000  # number of rows; small enough to train in seconds, large enough for a fair AUC

# --- Define the category vocabularies (strings, as they'd arrive in real data) ---
cities  = ["NYC", "LA", "CHI", "HOU", "PHX", "SEA", "MIA", "BOS"]  # cardinality 8
devices = ["ios", "android", "web", "tablet"]                       # cardinality 4
plans   = ["free", "basic", "plus", "pro", "enterprise"]            # cardinality 5

# Sample each categorical column i.i.d. uniformly over its vocabulary -> shape (N,)
city_col   = rng.choice(cities,  size=N)
device_col = rng.choice(devices, size=N)
plan_col   = rng.choice(plans,   size=N)

# Sample numeric columns from plausible distributions -> each shape (N,)
age     = rng.normal(40, 12, size=N).clip(18, 90)   # customer age
tenure  = rng.exponential(24, size=N).clip(0, 120)  # months as a customer
balance = rng.normal(0, 1, size=N)                  # already-standardized account balance

# --- Hidden 'true' latent weight for every category value (the signal to be learned) ---
# Different values -> different churn propensity. A model must distinguish categories to win.
city_w   = {c: w for c, w in zip(cities,  rng.normal(0, 1.2, len(cities)))}
device_w = {d: w for d, w in zip(devices, rng.normal(0, 1.0, len(devices)))}
plan_w   = {p: w for p, w in zip(plans,   rng.normal(0, 1.4, len(plans)))}

# Look up each row's latent categorical weight -> shape (N,)
z_city   = np.array([city_w[c]   for c in city_col])
z_device = np.array([device_w[d] for d in device_col])
z_plan   = np.array([plan_w[p]   for p in plan_col])

# Numeric contribution: standardize age/tenure first so the coefficients are comparable.
age_s    = (age - age.mean()) / age.std()
tenure_s = (tenure - tenure.mean()) / tenure.std()
z_num = 0.8 * age_s - 1.1 * tenure_s + 0.6 * balance

# Interaction term: the effect of 'age' DEPENDS on the city weight. This cross-term is what
# rewards models that combine categorical + numeric signal instead of treating them separately.
z_interaction = 0.9 * z_city * age_s

# Full logit + a little label noise, then draw Bernoulli labels.
logit = z_city + z_device + z_plan + z_num + z_interaction + rng.normal(0, 0.5, size=N)
prob  = 1.0 / (1.0 + np.exp(-logit))         # sigmoid -> churn probability in (0,1)
churn = (rng.random(N) < prob).astype(int)   # 1 with probability `prob`, else 0

# Assemble the dataframe exactly as messy real data would look (strings + floats).
df = pd.DataFrame({
    "city": city_col, "device": device_col, "plan": plan_col,
    "age": age, "tenure": tenure, "balance": balance,
    "churn": churn,
})

print(df.shape)
print("churn rate:", df['churn'].mean().round(3))  # sanity check: not wildly imbalanced
df.head()

## 2. Integer-encode the categoricals

An `nn.Embedding` is just a lookup table: it maps an **integer index** `0..C-1` to a learned row vector. So each categorical column must first be turned into contiguous integer codes. We use pandas' `category` dtype, whose `.cat.codes` gives exactly that — and we **store the cardinality** $C$ of every column, because the embedding table needs `num_embeddings = C`.

Numeric columns are standardized (zero mean, unit variance) using **train statistics only**, to help the neural net optimize and to avoid leakage.

In [ ]:
cat_cols = ["city", "device", "plan"]   # categorical feature names
num_cols = ["age", "tenure", "balance"] # numeric feature names
target   = "churn"

# Integer-encode each categorical to contiguous codes 0..C-1 and record its cardinality.
cardinalities = {}                       # column name -> number of distinct categories C
df_enc = df.copy()
for col in cat_cols:
    codes = df_enc[col].astype("category").cat.codes  # string -> int index, shape (N,)
    df_enc[col] = codes.astype(np.int64)              # embeddings need int64 (long) indices
    cardinalities[col] = int(df_enc[col].nunique())   # C for this column's embedding table

print("cardinalities:", cardinalities)   # e.g. {'city': 8, 'device': 4, 'plan': 5}

# --- Split into train / val / test (60 / 20 / 20), stratified on the target ---
X = df_enc[cat_cols + num_cols]
y = df_enc[target].values.astype(np.float32)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.40, random_state=SEED, stratify=y)          # 60% train
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)  # 20% val, 20% test

# Standardize numeric columns: fit the scaler on TRAIN ONLY, then apply to val/test.
scaler = StandardScaler().fit(X_tr[num_cols])
for part in (X_tr, X_val, X_te):
    part[num_cols] = scaler.transform(part[num_cols])   # in-place scaling of the 3 numeric cols

print(f"train={X_tr.shape}  val={X_val.shape}  test={X_te.shape}")

## 3. The embedding network

For a categorical column with cardinality $C$, we choose an embedding dimension with a common **rule of thumb** (from the fast.ai / Rossmann work):

$$d = \min\!\big(50,\; \lfloor (C+1)/2 \rfloor\big).$$

Small vocabularies get tiny vectors; large ones are capped at 50 so the table doesn't explode. Here `city` (C=8) → d=4, `device` (C=4) → d=2, `plan` (C=5) → d=3.

**Forward pass shapes** (batch size $B$):

```
x_cat  : (B, 3) int indices ── Embedding per column ──▶ [(B,4),(B,2),(B,3)] ──cat──▶ (B, 9)
x_num  : (B, 3) floats ─────────────────────────────────────────────────────────▶ (B, 3)
                              concat ▶ (B, 9+3=12) ▶ Linear+ReLU+Dropout ▶ ... ▶ (B, 1) logit
```

The final layer emits a single **logit** (no sigmoid) because we train with `BCEWithLogitsLoss`, which applies the sigmoid internally in a numerically stable way.

In [ ]:
def emb_dim(cardinality):
    """Rule-of-thumb embedding size: min(50, (C+1)//2)."""
    return min(50, (cardinality + 1) // 2)


class TabularEmbeddingNet(nn.Module):
    def __init__(self, cardinalities, n_numeric, hidden=(64, 32), p_drop=0.3):
        super().__init__()
        # One embedding table per categorical column. num_embeddings=C, embedding_dim=d.
        # nn.ModuleList registers them so their parameters are trained.
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=C, embedding_dim=emb_dim(C))
            for C in cardinalities
        ])
        emb_total = sum(emb_dim(C) for C in cardinalities)  # total width after concatenating embeddings
        in_features = emb_total + n_numeric                 # + the raw numeric columns

        # MLP head: Linear -> ReLU -> Dropout, stacked, then a final Linear to ONE logit.
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(p_drop)]
            prev = h
        layers += [nn.Linear(prev, 1)]   # output layer: (B, prev) -> (B, 1) raw logit
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        # x_cat: (B, n_cat) int64 indices ; x_num: (B, n_numeric) floats
        # Look up column j's embedding -> (B, d_j). emb(x_cat[:, j]) indexes the table row-wise.
        emb = [emb_layer(x_cat[:, j]) for j, emb_layer in enumerate(self.embeddings)]
        emb = torch.cat(emb, dim=1)          # concat along features -> (B, sum d_j)
        x = torch.cat([emb, x_num], dim=1)   # append numerics -> (B, sum d_j + n_numeric)
        return self.mlp(x).squeeze(1)        # (B, 1) -> (B,) so it matches the (B,) targets


cards = [cardinalities[c] for c in cat_cols]   # [8, 4, 5] in cat_cols order
model = TabularEmbeddingNet(cards, n_numeric=len(num_cols)).to(device)
print(model)
print("embedding dims:", {c: emb_dim(cardinalities[c]) for c in cat_cols})
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Training loop

We wrap the tensors in `TensorDataset` / `DataLoader` for mini-batching, optimize with **Adam**, and use **`BCEWithLogitsLoss`** (sigmoid + binary cross-entropy fused for stability) on the raw logits. A handful of epochs is enough on this small problem; we print train/val loss and validation **AUC** each epoch.

In [ ]:
def to_tensors(Xpart, ypart):
    """Split a dataframe part into an int64 categorical tensor, a float numeric tensor, and a float target."""
    x_cat = torch.tensor(Xpart[cat_cols].values, dtype=torch.long)     # (n, n_cat) indices for Embedding
    x_num = torch.tensor(Xpart[num_cols].values, dtype=torch.float32)  # (n, n_numeric) scaled floats
    y_t   = torch.tensor(ypart, dtype=torch.float32)                   # (n,) 0/1 targets
    return x_cat, x_num, y_t

xc_tr, xn_tr, yt_tr    = to_tensors(X_tr,  y_tr)
xc_val, xn_val, yt_val = to_tensors(X_val, y_val)
xc_te, xn_te, yt_te    = to_tensors(X_te,  y_te)

# DataLoader shuffles + batches the training set each epoch; val is evaluated in one shot.
train_loader = DataLoader(TensorDataset(xc_tr, xn_tr, yt_tr),
                          batch_size=512, shuffle=True)

criterion = nn.BCEWithLogitsLoss()                      # sigmoid + BCE on logits, stable
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

EPOCHS = 12
for epoch in range(1, EPOCHS + 1):
    # ---- training pass ----
    model.train()                                       # enable dropout
    running = 0.0
    for xb_cat, xb_num, yb in train_loader:
        optimizer.zero_grad()                           # clear gradients from last step
        logits = model(xb_cat, xb_num)                  # (B,) raw scores
        loss = criterion(logits, yb)                    # scalar BCE-with-logits loss
        loss.backward()                                 # backprop
        optimizer.step()                                # Adam parameter update
        running += loss.item() * len(yb)                # weight by batch size for a true mean
    train_loss = running / len(yt_tr)

    # ---- validation pass (no grad, no dropout) ----
    model.eval()
    with torch.no_grad():
        val_logits = model(xc_val, xn_val)              # (n_val,)
        val_loss = criterion(val_logits, yt_val).item()
        val_prob = torch.sigmoid(val_logits).numpy()    # logits -> probabilities for AUC
        val_auc = roc_auc_score(y_val, val_prob)
    print(f"epoch {epoch:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_auc {val_auc:.4f}")

### Neural-net test metrics

Evaluate the trained network once on the held-out **test** set and record accuracy + ROC-AUC for the final comparison.

In [ ]:
model.eval()
with torch.no_grad():
    test_prob_nn = torch.sigmoid(model(xc_te, xn_te)).numpy()  # (n_test,) churn probabilities
test_pred_nn = (test_prob_nn >= 0.5).astype(int)               # threshold at 0.5 -> labels

nn_acc = accuracy_score(y_te, test_pred_nn)
nn_auc = roc_auc_score(y_te, test_prob_nn)
print(f"Embedding NN  ->  test accuracy {nn_acc:.4f} | test roc_auc {nn_auc:.4f}")

## 5. Gradient-boosting baseline (one-hot features)

The standard non-neural approach: **one-hot encode** the categoricals and feed the resulting sparse matrix to a gradient-boosted tree ensemble. We use scikit-learn's `GradientBoostingClassifier`.

One-hot turns our 3 categorical columns (8 + 4 + 5 = 17 categories) into 17 binary columns — plus the 3 numerics = 20 features. With small cardinalities like these, one-hot is perfectly fine; the pain shows up when a column has thousands of categories.

> In production you'd reach for **XGBoost** or **LightGBM** here — same idea, dramatically faster, and they can consume categoricals natively. `GradientBoostingClassifier` is the closest baseline available offline in this environment.

In [ ]:
# One-hot encode categoricals with pandas. Fit the column layout on train, then reindex
# val/test to the SAME columns (fill missing category columns with 0) so shapes line up.
def one_hot(Xpart, columns=None):
    oh = pd.get_dummies(Xpart, columns=cat_cols)   # expands each cat col into indicator cols
    if columns is not None:
        oh = oh.reindex(columns=columns, fill_value=0)  # align to train's column set
    return oh

# NOTE: X_tr's cat columns are int codes; get_dummies still one-hots them per distinct code.
X_tr_oh = one_hot(X_tr)
oh_cols = X_tr_oh.columns                       # canonical feature layout learned on train
X_te_oh = one_hot(X_te, columns=oh_cols)        # same 20 columns as train

print("one-hot feature count:", X_tr_oh.shape[1])

gbm = GradientBoostingClassifier(random_state=SEED)   # sensible defaults; fast on 20 features
gbm.fit(X_tr_oh, y_tr)

test_prob_gbm = gbm.predict_proba(X_te_oh)[:, 1]      # P(churn=1) for each test row
test_pred_gbm = (test_prob_gbm >= 0.5).astype(int)

gbm_acc = accuracy_score(y_te, test_pred_gbm)
gbm_auc = roc_auc_score(y_te, test_prob_gbm)
print(f"GradientBoosting ->  test accuracy {gbm_acc:.4f} | test roc_auc {gbm_auc:.4f}")

## 6. Head-to-head

Both models on the same held-out test set, side by side (printed table + bar chart).

In [ ]:
# Collect the four numbers into a tidy dataframe for a clean printout.
results = pd.DataFrame({
    "model":    ["Embedding NN", "GradientBoosting"],
    "accuracy": [nn_acc, gbm_acc],
    "roc_auc":  [nn_auc, gbm_auc],
})
print(results.to_string(index=False))

# Grouped bar chart: accuracy and ROC-AUC per model.
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(results))          # bar group positions: [0, 1]
w = 0.35                             # bar width
b1 = ax.bar(x - w/2, results["accuracy"], w, label="accuracy", color="#6699cc")
b2 = ax.bar(x + w/2, results["roc_auc"],  w, label="roc_auc",  color="#cc6666")
ax.set_xticks(x); ax.set_xticklabels(results["model"])
ax.set_ylim(0, 1.0); ax.set_ylabel("score"); ax.set_title("NN embeddings vs Gradient Boosting (test set)")
ax.legend()
for bars in (b1, b2):                 # annotate each bar with its value
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Discussion — embeddings vs one-hot, and when GBMs still win

### Why learned embeddings beat one-hot for high-cardinality categoricals

- **Dense, not sparse.** One-hot spends one column per category, so a feature with 50,000 values (zip codes, product IDs, user IDs) becomes 50,000 mostly-zero columns. An embedding replaces that with a single dense vector of, say, 50 numbers — orders of magnitude fewer inputs.
- **Fewer parameters downstream.** With one-hot, the first dense layer needs a weight for every one of those 50,000 columns. With an embedding, the table itself is `50,000 × 50`, and the layer that follows only sees 50 inputs.
- **They capture similarity / geometry.** One-hot makes every category exactly equidistant (orthogonal) — 'NYC' is as unrelated to 'BOS' as it is to 'PHX'. A learned embedding places *behaviourally similar* categories near each other because the loss gradient pulls them together. That structure transfers: rare categories borrow signal from their neighbours.
- **Reusable representations.** The trained vectors can be exported and reused as features in other models — a genuine learned encoding of the entity.

### The embedding-dimension heuristic

$$d = \min\!\big(50,\ \lfloor (C+1)/2 \rfloor\big)$$

Bigger vocabularies get a bit more room to express structure, but the `min(50, ...)` cap keeps the table (and its parameter count) bounded even for huge cardinalities. It's only a starting point — treat `d` as a hyperparameter to tune.

### When gradient-boosted trees still win on tabular data

- **Small / medium datasets.** GBMs are extremely sample-efficient and need almost no tuning to get a strong result; neural nets are hungrier for data.
- **Mostly-numeric, low-cardinality data.** If there are few categories, one-hot is cheap and trees split on thresholds naturally — the embedding advantage largely disappears (as you can see in the scores above, the two are close on this easy synthetic problem).
- **Speed, robustness, and interpretability.** GBMs train fast on CPU, are insensitive to feature scaling and monotone transforms, handle missing values, and offer feature importances / SHAP out of the box.
- **Strong, low-effort baselines.** On many real tabular benchmarks, **XGBoost / LightGBM** still match or beat deep nets. The embedding-net approach shines when cardinalities are high, the data is large, or you want to fold tabular features into a larger deep model (e.g. alongside text or images).

**Bottom line:** always benchmark both. Reach for entity embeddings when categorical cardinality is high or when the tabular features live inside a bigger neural pipeline; reach for gradient boosting (ideally XGBoost/LightGBM) as the default strong baseline everywhere else.